# 🛡️ Trust-Aware Machine Learning for Credit Card Fraud Detection
### *LUMS MS AI Semester Project / Research Paper*

> **Core Question**: *Why does a model with 0.9924 ROC-AUC still produce catastrophic, overconfident mistakes in production—and how can we make it trust-aware?*

---
### 📚 Theoretical Framework:
1. **Calibration**: Guo et al. (ICML 2017) *On Calibration of Modern Neural Networks*
2. **Epistemic Uncertainty**: Gal & Ghahramani (ICML 2016) *Dropout as a Bayesian Approximation*
3. **Selective Classification**: Geifman & El-Yaniv (NeurIPS 2017) *Selective Classification for Deep Neural Networks*
4. **Safety & Robustness**: Hendrycks & Gimpel (ICLR 2017), Amodei et al. (2016)

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import load_fraud_data, prepare_datasets, create_dataloaders
from src.models.credit_net import CreditNet
from src.models.trainer import train_model, evaluate_predictions
from src.calibration.metrics import binary_expected_calibration_error, class_conditional_ece, brier_score_loss, reliability_diagram_data
from src.calibration.temperature import fit_temperature_scaling
from src.uncertainty.mc_dropout import mc_dropout_predict, compute_uncertainty_metrics, identify_failure_modes
from src.selective.policy import DualGatedTrustPolicy
from src.selective.risk_coverage import FinancialCostMatrix, compute_risk_coverage_curve, compute_cost_coverage_curve
from src.visualization.plots import plot_reliability_diagram, plot_risk_coverage_curves, plot_uncertainty_distributions, plot_cost_coverage_curve

print("PyTorch Version:", torch.__version__)
print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")

## 1. Data Pipeline: Imbalance & Controlled Splitting
In financial fraud detection, random 80/20 train/test splits risk data leakage and hide calibration drift.
We implement a strict 3-way split: **Train (60%)**, **Validation (20%)**, and **Test (20%)**.

In [ ]:
# Load Kaggle credit card dataset
df = load_fraud_data("../data/creditcard.csv")
print(f"Total transactions: {len(df):,} | Fraudulent: {int(df['Class'].sum()):,} ({df['Class'].mean()*100:.3f}%)")

# Prepare 3-way split
data = prepare_datasets(df, split_strategy="stratified", val_size=0.2, test_size=0.2, random_state=42)
loaders = create_dataloaders(data, batch_size=512)

train_neg = np.sum(data["y_train"] == 0)
train_pos = max(1, np.sum(data["y_train"] == 1))
pos_weight = float(train_neg / train_pos)
print(f"Calculated positive class weight for loss: {pos_weight:.2f}")

## 2. Train Backbone Architecture: CreditNet
We use  with Dropout layers (=0.3$) that serve a dual purpose:
- Regularization during training
- Approximate Bayesian inference (Monte Carlo Dropout) at test time

In [ ]:
model = CreditNet(input_dim=data["X_train"].shape[1], dropout_rate=0.3)
train_res = train_model(
    model=model,
    train_loader=loaders["train"],
    val_loader=loaders["val"],
    pos_weight=pos_weight,
    epochs=15,
    lr=1e-3,
    verbose=True
)

## 3. The Miscalibration Paradox & Post-Hoc Temperature Scaling
A model can achieve **ROC-AUC > 0.99** while being dangerously uncalibrated.
We optimize scalar $ on the independent validation split using L-BFGS (Guo et al. 2017).

In [ ]:
model.eval()
with torch.no_grad():
    val_logits = model(torch.tensor(data["X_val"], dtype=torch.float32)).numpy().ravel()
    test_logits = model(torch.tensor(data["X_test"], dtype=torch.float32)).numpy().ravel()

test_probs_uncal = torch.sigmoid(torch.tensor(test_logits)).numpy()

# Fit Temperature Scaler on Validation set ONLY (no test leakage)
scaler, opt_T = fit_temperature_scaling(val_logits, data["y_val"])
print(f"Learned Optimal Temperature T: {opt_T:.4f}")

with torch.no_grad():
    test_logits_scaled = scaler(torch.tensor(test_logits)).numpy().ravel()
    test_probs_cal = torch.sigmoid(torch.tensor(test_logits_scaled)).numpy()

# Compute corrected Binary ECE and Class-Conditional ECE
ece_uncal = binary_expected_calibration_error(data["y_test"], test_probs_uncal)
ece_cal = binary_expected_calibration_error(data["y_test"], test_probs_cal)
brier_uncal = brier_score_loss(data["y_test"], test_probs_uncal)
brier_cal = brier_score_loss(data["y_test"], test_probs_cal)

print(f"ECE Before: {ece_uncal:.4f}  -->  ECE After: {ece_cal:.4f}")
print(f"Brier Before: {brier_uncal:.4f} --> Brier After: {brier_cal:.4f}")

In [ ]:
# Plot Reliability Diagram
rel_uncal = reliability_diagram_data(data["y_test"], test_probs_uncal)
rel_cal = reliability_diagram_data(data["y_test"], test_probs_cal)

fig_rel = plot_reliability_diagram(rel_uncal, rel_cal, ece_uncal, ece_cal)
plt.show()

## 4. Epistemic Uncertainty via Monte Carlo Dropout
We execute =50$ stochastic forward passes with active dropout.
This provides the predictive posterior standard deviation $\sigma_{	ext{MC}}$ to capture model uncertainty.

In [ ]:
mc_res = mc_dropout_predict(
    model=model,
    X_tensor=torch.tensor(data["X_test"], dtype=torch.float32),
    n_samples=50
)

failures = identify_failure_modes(
    y_true=data["y_test"],
    probs=mc_res["mean_probs"],
    uncertainty=mc_res["std_epistemic"],
    conf_threshold=0.90,
    uncertainty_threshold=0.10
)

print(f"Total Errors: {failures['total_errors']}")
print(f"High-Confidence Errors: {failures['high_conf_wrong_count']}")
print(f"Recoverable via Uncertainty (> 0.10 std): {failures['recoverable_error_count']}")
print(f"Silent Killers (High Conf, Low Unc): {failures['silent_killer_count']}")

## 5. Cost-Sensitive Selective Classification (Human-in-the-Loop)
In automated banking systems, transactions where the model is uncertain are deferred to Tier-2 human fraud auditors.
We model financial utility using a realistic cost matrix:  = 50,  = 5, $	ext{Audit} = .

In [ ]:
cost_matrix = FinancialCostMatrix(cost_fn=250.0, cost_fp=15.0, cost_audit=5.0)
risk_cov = compute_risk_coverage_curve(data["y_test"], test_probs_cal, mc_res["std_epistemic"])
cost_cov = compute_cost_coverage_curve(data["y_test"], test_probs_cal, mc_res["std_epistemic"], cost_matrix)

det_preds = (test_probs_cal >= 0.5).astype(int)
base_cost = cost_matrix.compute_total_cost(data["y_test"], det_preds, np.zeros(len(data["y_test"]), dtype=bool))

fig_rc = plot_risk_coverage_curves(risk_cov)
plt.show()

fig_cost = plot_cost_coverage_curve(cost_cov, baseline_cost=base_cost["total_cost"])
plt.show()